#### 🥉 Pipeline de Ingestão - Camada Bronze (Bruta)

##### Objetivo
Este notebook implementa o processo de **Ingestão de Dados na Camada Bronze**, realizando:

* **Carregamento** de arquivos CSV da camada de origem (Volume)
* **Captura de Metadados** de arquivos (caminho e data de modificação)
* **Persistência** incremental na tabela `tb_bronze`
* **Preparação** para processamento nas camadas superiores (Silver/Gold)

##### Fluxo de Dados
```
/Volumes/vendas_pecas/default/my/processar/ → tb_raw_data
```

##### Tecnologias Utilizadas
* **COPY INTO**: Carregamento batch incremental de arquivos CSV
* **Delta Lake**: Formato de armazenamento com suporte a ACID e time travel

##### Catalog e Schema
* **Catalog**: `vendas_pecas`
* **Schema**: `camada_bronze`

In [0]:
USE vendas_pecas.camada_bronze;

#### 1️⃣ Carregamento Incremental

##### Objetivo
Carregar arquivos CSV do Volume de origem para a tabela Bronze de forma incremental e idempotente.

##### Estratégia: COPY INTO
* `Idempotência`: Não reprocessa arquivos já carregados (rastreamento automático)
* `Fonte`: `/Volumes/vendas_pecas/default/my/processar/`
* `Destino`: `tb_raw_data`

##### Metadados Capturados
* `_metadata.file_path`: Caminho completo do arquivo de origem
* `_metadata.file_modification_time`: Data/hora de modificação do arquivo (usado como `ingest_date`)

##### Recursos Utilizados
* **FILEFORMAT = CSV**: Especifica o formato dos arquivos de entrada
* **FORMAT_OPTIONS**:
  * `inferSchema = false`: Desabilita inferência automática (usa schema da tabela de destino)
  * `mergeSchema = false`: Não mescla schemas diferentes
  * `header = true`: Primeira linha contém nomes de colunas
  * `delimiter = ','`: Separador de colunas
* **COPY_OPTIONS**:
  * `mergeSchema = false`: Mantém schema fixo da tabela

##### Output
Tabela Delta: `tb_raw_data` (append incremental)

In [0]:
COPY INTO
  tb_raw_data
FROM
  (
    SELECT 
      *,
      _metadata.file_path as file_path,
      _metadata.file_modification_time as ingest_date
    FROM 
      "/Volumes/vendas_pecas/default/my/processar/"
  )
  FILEFORMAT = CSV
  FORMAT_OPTIONS (
    'inferSchema' = 'false', 
    'mergeSchema' = 'false',
    'header' = 'true',
    'delimiter' = ','
  )
  COPY_OPTIONS(
    'mergeSchema' = 'false'
  )

In [0]:
-- CREATE OR REFRESH STREAMING TABLE tb_arquivos_bronze
-- AS SELECT *, 
-- _metadata.file_path as file_path,
-- _metadata.file_modification_time as ingest_date
-- FROM STREAM read_files(
--   "/Volumes/vendas_pecas/default/my/processar/",
--   format => "csv",
--   delimiter => ",",
--   header => "true",
--   inferColumnTypes => "false",
--   schemaEvolutionMode => "rescue"
-- );

In [0]:
-- MERGE INTO 
--  tb_bronze tb
-- USING 
--   (
--     SELECT *
--     FROM (
--       SELECT *, 
--         ROW_NUMBER() OVER(
--             PARTITION BY id_nf
--             ORDER BY ingest_date DESC
--         ) AS row_num
--       FROM tb_arquivos_bronze
--     )
--     WHERE 
--       row_num = 1
--   ) AS source_table 
-- ON 
--   tb.id_nf = source_table.id_nf
-- WHEN 
--   MATCHED
--   AND source_table.ingest_date > tb.ingest_date 
-- THEN 
--   UPDATE SET *
-- WHEN
--  NOT MATCHED 
-- THEN
--   INSERT *